# India Job Market Intelligence Platform

# Phase 7: SQL Business Analysis

---

## Objective

The objective of this phase is to analyze the Indian job market using SQL and generate actionable business insights for recruiters, hiring managers, and business leaders.

The analysis is performed on the dimensional data warehouse developed in the previous phase.

---

## Technology Stack

- MySQL
- SQL
- Jupyter Notebook
- SQLAlchemy
- Pandas

---

## Analysis Categories

- Executive Hiring Overview
- Company Hiring Analysis
- Geographic Hiring Analysis
- Salary Intelligence
- Experience Analysis
- Skills Analysis
- Recruitment Trends

In [1]:
# importing libraries
import pandas as pd

from sqlalchemy import create_engine
from sqlalchemy import text

In [3]:
# connect to mysql
MYSQL_USER = "root"
MYSQL_PASSWORD = "27104720A"

MYSQL_HOST = "localhost"

MYSQL_PORT = 3306

MYSQL_DATABASE = "job_market_analytics"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("Database Connected Successfully")

Database Connected Successfully


# Executive Hiring Overview

## Business Question 1

### How many job postings are available in the warehouse?

This KPI provides an overview of the recruitment market size represented in the dataset.

In [9]:
pd.read_sql("""
SELECT COUNT(*) AS Total_Job_Postings
FROM fact_jobs;
""", engine)

,Total_Job_Postings
0,23201


### Business Insight

This metric represents the total number of unique job postings available for analysis.

# Business Question 2
# Number of Companies Hiring

In [16]:
pd.read_sql("""
SELECT COUNT(*) AS Total_Companies
FROM dim_company;
""", engine)

,Total_Companies
0,7022


# Business Question 3
# Number of Cities

In [19]:
pd.read_sql("""
SELECT COUNT(DISTINCT primary_city) AS Total_Cities
FROM dim_location;
""", engine)

,Total_Cities
0,198


# Business Question 4
# Average Salary Offered

In [22]:
pd.read_sql("""
SELECT
ROUND(AVG(salary_midpoint_lpa),2)
AS Average_Salary_LPA
FROM fact_jobs
WHERE salary_midpoint_lpa IS NOT NULL;
""", engine)

,Average_Salary_LPA
0,1.83


# Business Question 5
# Average Experience Required

In [25]:
pd.read_sql("""
SELECT
ROUND(AVG(experience_min_yrs),2)
AS Average_Minimum_Experience
FROM fact_jobs;
""", engine)

,Average_Minimum_Experience
0,4.14


# Business Question 6
# Remote vs Hybrid vs On-site

In [28]:
pd.read_sql("""
SELECT

r.work_mode,

COUNT(*) AS Total_Jobs

FROM fact_jobs f

JOIN dim_role r

ON f.role_key=r.role_key

GROUP BY r.work_mode

ORDER BY Total_Jobs DESC;
""", engine)

,work_mode,Total_Jobs
0,On-site,18701
1,Hybrid,2347
2,Remote,2153


# Company Hiring Analysis

## Objective

Analyze hiring behavior across companies to identify the most active recruiters, salary trends, hiring preferences, and recruitment patterns.

These insights help understand employer demand and recruitment strategies within the Indian job market.

# Business Question 7
# Which companies have posted the highest number of jobs?

In [32]:
pd.read_sql("""
SELECT
    c.company_name,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
GROUP BY c.company_name
ORDER BY total_jobs DESC
LIMIT 15;
""", engine)

,company_name,total_jobs
0,Tata Consultancy Services,654
1,Accenture,438
2,EY,340
3,Leading Client,244
4,Infosys,209
5,Capgemini,195
6,Photon,193
7,Logic Planet,185
8,Hexaware Technologies,147
9,Sparix Global,126


The analysis identifies the companies with the highest hiring activity. These organizations represent major employers in the data job market and may offer the greatest number of opportunities for job seekers.

# Business Question 8
# Which companies offer the highest average salary?

In [36]:
pd.read_sql("""
SELECT
    c.company_name,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary_lpa
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
WHERE f.salary_midpoint_lpa IS NOT NULL
GROUP BY c.company_name
HAVING COUNT(*) >= 5
ORDER BY avg_salary_lpa DESC
LIMIT 15;
""", engine)

,company_name,avg_salary_lpa
0,Fortune 500 IT MNC,40.00
1,Wize Careers Consultants,28.70
2,US MNC (analytics),28.67
3,Leading IT Software Company,26.69
4,India's Largest IT Service Provider,25.00
5,Sunovaa Tech,24.60
6,Turing Global India,24.50
7,KPI Partners,23.54
8,Black and white Business Solution,22.55
9,client of kaizen,22.42


# Business Question 9
# Which companies have the most remote job opportunities?

In [46]:
pd.read_sql("""
SELECT
    c.company_name,
    COUNT(*) AS remote_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE r.work_mode = 'Remote'
GROUP BY c.company_name
ORDER BY remote_jobs DESC
LIMIT 15;
""", engine)

,company_name,remote_jobs
0,Sparix Global,60
1,Nagarro,53
2,Welo Data,52
3,Logic Planet,45
4,Infraveo Technologies,38
5,Highlevel Llc,35
6,Codvo,34
7,Airo Digital Labs,28
8,Numerator,24
9,Exavalu,23


## Section Summary

Key observations from the company analysis include:

- The most active hiring organizations.
- Companies offering higher average salaries.
- Employers with significant fresher hiring.
- Organizations recruiting experienced professionals.
- Companies providing the largest number of remote opportunities.

These insights help job seekers prioritize employers based on career goals and hiring patterns.

# Geographic Hiring Analysis

## Objective

Analyze the geographical distribution of job opportunities across Indian cities to identify hiring hotspots, salary variations, remote work adoption, and regional recruitment trends.

These insights assist recruiters in workforce planning and help job seekers identify cities with strong employment opportunities.

# Business Question 10
# Which cities have the highest number of job postings?

Why this matters

Understanding hiring hotspots helps companies plan recruitment campaigns and helps candidates identify cities with the strongest employment opportunities.

In [52]:
pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
GROUP BY l.primary_city
ORDER BY total_jobs DESC
LIMIT 15;
""", engine)

,primary_city,total_jobs
0,Mumbai,3276
1,Bangalore,2830
2,Chennai,2666
3,Pune,2606
4,Noida,2585
5,Gurgaon,2210
6,Remote,2010
7,Kolkata,1735
8,Ahmedabad,1352
9,Delhi,920


Cities with the highest number of postings represent major technology and business hubs where demand for analytics professionals is concentrated.

# Business Question 11
# Which cities offer the highest average salary?

In [56]:
pd.read_sql("""
SELECT
    l.primary_city,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
WHERE f.salary_midpoint_lpa IS NOT NULL
GROUP BY l.primary_city
HAVING COUNT(*) >= 10
ORDER BY avg_salary DESC
LIMIT 15;
""", engine)

,primary_city,avg_salary
0,Pune(Baner),8.06
1,Pune(Hinjewadi Phase 1),6.23
2,Hyderabad,5.35
3,Mumbai Suburban,2.61
4,Chennai,2.37
5,Pune,2.35
6,Delhi,1.94
7,Noida,1.74
8,Mumbai,1.73
9,Gurgaon,1.46


# Business Question 12
# Which cities have the most remote opportunities?

In [65]:
pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(*) AS remote_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE r.work_mode = 'Remote'
GROUP BY l.primary_city
ORDER BY remote_jobs DESC
LIMIT 15;
""", engine)

,primary_city,remote_jobs
0,Remote,2010
1,Mumbai,23
2,Bangalore,18
3,Pune,18
4,Gurgaon,16
5,Kolkata,15
6,Delhi,15
7,Chennai,15
8,Noida,12
9,Ahmedabad,5


In [77]:
pd.read_sql("""
SELECT
    primary_city,
    work_mode,
    COUNT(*) AS total_jobs
FROM jobs_final
WHERE primary_city = 'Remote'
GROUP BY primary_city, work_mode;
""", engine)

,primary_city,work_mode,total_jobs
0,Remote,Remote,2010


In [79]:
pd.read_sql("""
SELECT
    location,
    primary_city,
    work_mode
FROM jobs_final
WHERE primary_city = 'Remote'
LIMIT 20;
""", engine)

,location,primary_city,work_mode
0,Remote,Remote,Remote
1,Remote,Remote,Remote
2,Remote,Remote,Remote
3,Remote,Remote,Remote
4,Remote,Remote,Remote
5,Remote,Remote,Remote
6,Remote,Remote,Remote
7,Remote,Remote,Remote
8,Remote,Remote,Remote
9,Remote,Remote,Remote


# Which cities (excluding fully remote postings) offer the highest number of remote-enabled jobs?

In [82]:
pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(*) AS remote_jobs
FROM fact_jobs f
JOIN dim_location l
    ON f.location_key = l.location_key
JOIN dim_role r
    ON f.role_key = r.role_key
WHERE r.work_mode = 'Remote'
  AND l.primary_city <> 'Remote'
GROUP BY l.primary_city
ORDER BY remote_jobs DESC
LIMIT 15;
""", engine)

,primary_city,remote_jobs
0,Mumbai,23
1,Bangalore,18
2,Pune,18
3,Gurgaon,16
4,Kolkata,15
5,Delhi,15
6,Chennai,15
7,Noida,12
8,Ahmedabad,5
9,Mumbai(Vikhroli),1


# Business Question 13
# Which cities hire the most freshers?

In [68]:
pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(*) AS fresher_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE r.is_fresher_friendly = 1
GROUP BY l.primary_city
ORDER BY fresher_jobs DESC
LIMIT 15;
""", engine)

,primary_city,fresher_jobs
0,Mumbai,614
1,Noida,545
2,Gurgaon,441
3,Kolkata,396
4,Remote,395
5,Chennai,386
6,Bangalore,379
7,Pune,368
8,Ahmedabad,343
9,Delhi,167


# Business Question 14
# Which cities require the highest average experience?

In [71]:
pd.read_sql("""
SELECT
    l.primary_city,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_experience
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
GROUP BY l.primary_city
ORDER BY avg_experience DESC
LIMIT 15;
""", engine)

,primary_city,avg_experience
0,Mumbai(Chandivali),10.00
1,Kolkata(Ep Block Sector 5 Salt Lake),10.00
2,Mumbai(Masjid Bunder),9.00
3,Mumbai(Wadala East),8.00
4,Kolkata(Minto Park),8.00
5,Kolkata(New Town +1),8.00
6,Ahmedabad(Ahmedabad Cantonment),8.00
7,Mysuru,8.00
8,Pune(Senapati Bapat Road),8.00
9,Pune(Viman Nagar +1),8.00


# Business Question 15
# Which cities have the highest concentration of Data Analyst roles?

In [74]:
pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(*) AS data_analyst_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE r.role_category = 'Data Analyst'
GROUP BY l.primary_city
ORDER BY data_analyst_jobs DESC
LIMIT 15;
""", engine)

,primary_city,data_analyst_jobs
0,Noida,606
1,Bangalore,541
2,Ahmedabad,532
3,Kolkata,512
4,Pune,498
5,Chennai,477
6,Mumbai,464
7,Gurgaon,416
8,Remote,360
9,Delhi,164


## Section Summary

The geographic analysis reveals the regional distribution of analytics jobs across India.

Key findings include:

- Cities with the highest hiring demand.
- Cities offering competitive salaries.
- Regions with stronger remote work adoption.
- Locations providing greater opportunities for fresh graduates.
- Cities with higher demand for experienced professionals.

These insights support recruitment planning, workforce allocation, and informed career decisions.

# Salary Intelligence

## Objective

Analyze salary trends across roles, locations, companies, and experience levels to identify compensation patterns in the Indian technology job market.

These insights help job seekers make informed career decisions and enable recruiters to benchmark competitive salary offerings.

# Business Question 16
# Which job roles offer the highest average salary?

In [6]:
salary_by_role = pd.read_sql("""
SELECT
    r.role_category,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary_lpa,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
WHERE f.salary_disclosed = 1
GROUP BY r.role_category
HAVING COUNT(*) >= 20
ORDER BY avg_salary_lpa DESC;
""", engine)

salary_by_role

,role_category,avg_salary_lpa,total_jobs
0,Data Scientist,18.59,757
1,Data Engineer,17.77,216
2,Machine Learning Engineer,16.29,473
3,Python Developer,15.72,222
4,Business Analyst,14.14,460
5,Data Analyst,10.62,640


Roles with higher average salaries indicate stronger market demand and greater business value associated with those skill sets.

# Business Question 17
# How does the average salary change with the minimum experience required?

In [17]:
salary_by_experience = pd.read_sql("""
SELECT
    experience_min_yrs,
    ROUND(AVG(salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS total_jobs
FROM fact_jobs
WHERE salary_disclosed = 1
GROUP BY experience_min_yrs
HAVING COUNT(*) >= 20
ORDER BY experience_min_yrs;
""", engine)

salary_by_experience

,experience_min_yrs,avg_salary,total_jobs
0,0,9.62,237
1,1,5.25,147
2,2,8.15,213
3,3,11.66,350
4,4,13.62,483
5,5,17.41,461
6,6,17.87,344
7,7,21.86,184
8,8,22.92,202
9,9,24.51,51


In [29]:
salary_summary = (
    salary_df
    .groupby("experience_min_yrs")["salary_midpoint_lpa"]
    .agg(
        avg_salary="mean",
        median_salary="median",
        total_jobs="count"
    )
    .round(2)
    .reset_index()
)

salary_summary

,experience_min_yrs,avg_salary,median_salary,total_jobs
0,0,9.62,4.50,237
1,1,5.25,4.25,147
2,2,8.15,6.50,213
3,3,11.66,11.00,350
4,4,13.62,14.00,483
5,5,17.41,17.50,461
6,6,17.87,17.50,344
7,7,21.86,22.50,184
8,8,22.92,22.50,202
9,9,24.51,27.50,51


## Data Validation Note

During salary analysis, an anomaly was observed where jobs requiring 0 years of experience had a higher average salary than some jobs requiring 1 year of experience.

Further investigation showed that salary data is positively skewed by a small number of premium fresher hiring programs. Therefore, median salary was calculated alongside average salary to provide a more representative measure of central tendency.

This validation step ensures that business conclusions are not based solely on averages influenced by outliers.

# Business Question 18
# Which cities offer the highest salaries?

In [33]:
salary_by_city = pd.read_sql("""
SELECT
    l.primary_city,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key=l.location_key
WHERE
    f.salary_disclosed = 1
    AND l.primary_city <> 'Remote'
GROUP BY l.primary_city
HAVING COUNT(*) >= 10
ORDER BY avg_salary DESC
LIMIT 15;
""", engine)

salary_by_city

,primary_city,avg_salary,total_jobs
0,Hyderabad,19.86,145
1,Pune,19.04,322
2,Chennai,17.02,371
3,Mumbai,16.07,353
4,Delhi,15.90,112
5,Bangalore,15.75,261
6,Gurgaon,14.85,217
7,Pune(Baner),13.63,13
8,Noida,13.55,332
9,Kolkata,12.73,154


# Business Question 19
# Which companies offer the highest average salary?

In [38]:
salary_by_company = pd.read_sql("""
SELECT
    c.company_name,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key=c.company_key
WHERE f.salary_disclosed = 1
GROUP BY c.company_name
HAVING COUNT(*) >= 5
ORDER BY avg_salary DESC
LIMIT 15;
""", engine)

salary_by_company

,company_name,avg_salary,total_jobs
0,Fortune 500 IT MNC,40.00,8
1,Optum,35.11,11
2,KPI Partners,31.95,14
3,EXL,28.86,11
4,Wize Careers Consultants,28.70,5
5,US MNC (analytics),28.67,61
6,eLabs Infotech,28.54,6
7,Calsoft,27.83,6
8,Kiash Solutions LLP,27.21,7
9,Leading IT Software Company,26.69,8


# Business Question 20
# Which companies offer the highest salary for Data Analysts?

In [46]:
pd.read_sql("""
SELECT
    c.company_name,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE
    r.role_category = 'Data Analyst'
    AND f.salary_disclosed = 1
GROUP BY c.company_name
HAVING COUNT(*) >= 3
ORDER BY avg_salary DESC
LIMIT 15;
""", engine)

,company_name,avg_salary,jobs
0,EXL,28.62,4
1,TechF,25.12,4
2,US MNC (analytics),24.63,17
3,EY,23.75,3
4,Dentsu Global Services,20.25,3
5,CGI,17.61,7
6,PURVIEW,16.33,3
7,Black and white Business Solution,14.38,4
8,Gigafactor Solutions,14.00,5
9,Cloud Counselage,14.00,3


# Business Question 21
# Which work mode offers the highest salaries?

In [43]:
salary_workmode = pd.read_sql("""
SELECT
    r.work_mode,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key=r.role_key
WHERE f.salary_disclosed = 1
GROUP BY r.work_mode
ORDER BY avg_salary DESC;
""", engine)

salary_workmode

,work_mode,avg_salary,total_jobs
0,Hybrid,21.01,721
1,Remote,14.49,166
2,On-site,13.21,1881


# Experience & Career Progression Analysis

## Objective

Analyze experience requirements across job roles, companies, locations, and work arrangements to understand employer hiring expectations and career progression opportunities in the Indian job market.

The analysis helps identify entry-level opportunities, senior hiring trends, and experience requirements across different segments of the technology industry.

# Business Question 22
# Which job roles require the highest minimum experience?

Why this matters

Different technology roles demand different levels of experience. Understanding these requirements helps candidates plan career progression and enables recruiters to benchmark hiring expectations.

In [51]:
experience_by_role = pd.read_sql("""
SELECT
    r.role_category,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_min_experience,
    ROUND(AVG(f.experience_max_yrs),2) AS avg_max_experience,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.role_category
ORDER BY avg_min_experience DESC;
""", engine)

experience_by_role

,role_category,avg_min_experience,avg_max_experience,total_jobs
0,Data Engineer,4.80,8.71,1922
1,Data Scientist,4.47,8.26,6455
2,Machine Learning Engineer,4.30,8.03,4004
3,Business Analyst,4.12,7.76,4505
4,Python Developer,4.05,7.78,1586
5,Data Analyst,3.31,6.80,4729


# Business Question 23
# Which companies hire the most freshers?

In [55]:
fresher_companies = pd.read_sql("""
SELECT
    c.company_name,
    COUNT(*) AS fresher_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
WHERE f.experience_min_yrs = 0
GROUP BY c.company_name
ORDER BY fresher_jobs DESC
LIMIT 15;
""", engine)

fresher_companies

,company_name,fresher_jobs
0,Accenture,27
1,EY,20
2,Logic Planet,18
3,Leading Client,15
4,Nagarro,15
5,Iris Software,14
6,Tata Consultancy Services,14
7,Busigence Technologies,13
8,Bajaj Finance,13
9,Crisil,12


Business Recommendation

Fresh graduates should prioritize these organizations while planning their application strategy.

# Business Question 24
# Which companies primarily recruit experienced professionals?

In [59]:
senior_companies = pd.read_sql("""
SELECT
    c.company_name,
    COUNT(*) AS senior_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
WHERE f.experience_min_yrs >= 5
GROUP BY c.company_name
ORDER BY senior_jobs DESC
LIMIT 15;
""", engine)

senior_companies

,company_name,senior_jobs
0,Tata Consultancy Services,498
1,Accenture,210
2,Leading Client,149
3,Infosys,145
4,EY,115
5,Photon,112
6,Capgemini,109
7,Logic Planet,105
8,Hexaware Technologies,92
9,Sparix Global,88


# Business Question 25
# Which cities demand the most experienced professionals?

In [62]:
experienced_cities = pd.read_sql("""
SELECT
    l.primary_city,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_experience,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
WHERE l.primary_city <> 'Remote'
GROUP BY l.primary_city
HAVING COUNT(*) >= 10
ORDER BY avg_experience DESC;
""", engine)

experienced_cities

,primary_city,avg_experience,total_jobs
0,Hyderabad,5.16,538
1,Pune(Baner),5.05,22
2,Chennai,4.70,2666
3,Pune,4.45,2606
4,Bangalore,4.30,2830
5,Pune(Hinjewadi Phase 1),4.18,11
6,Noida,4.06,2585
7,Mumbai,3.94,3276
8,Delhi,3.94,920
9,Gurgaon,3.91,2210


# Business Question 26
# Which work mode is most common for entry-level jobs?

In [65]:
entry_workmode = pd.read_sql("""
SELECT
    r.work_mode,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
WHERE f.experience_min_yrs = 0
GROUP BY r.work_mode
ORDER BY total_jobs DESC;
""", engine)

entry_workmode

,work_mode,total_jobs
0,On-site,1910
1,Remote,208
2,Hybrid,101


This tells us whether freshers are more likely to find Remote, Hybrid, or On-site roles.

# Business Question 27
# Which roles have the widest experience range?

In [69]:
experience_flexibility = pd.read_sql("""
SELECT
    r.role_category,
    ROUND(AVG(f.experience_span),2) AS avg_experience_span,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.role_category
HAVING COUNT(*) >= 20
ORDER BY avg_experience_span DESC;
""", engine)

experience_flexibility

,role_category,avg_experience_span,total_jobs
0,Data Engineer,3.91,1922
1,Data Scientist,3.79,6455
2,Machine Learning Engineer,3.73,4004
3,Python Developer,3.72,1586
4,Business Analyst,3.63,4505
5,Data Analyst,3.49,4729


# Why this matters

## A larger experience span suggests employers are flexible about the level of candidate they are willing to consider, while a smaller span often indicates more specific hiring requirements.

## Section Summary

The experience analysis highlights employer expectations across roles, companies, cities, and work arrangements.

Key findings include:

- Roles requiring the highest experience.
- Companies actively hiring fresh graduates.
- Organizations focused on experienced professionals.
- Cities demanding experienced talent.
- Work modes most suitable for entry-level candidates.
- Roles with the most flexible experience requirements.

These insights help candidates plan career growth and support recruiters in benchmarking hiring strategies.

# Skills Intelligence

## Objective

Analyze the demand for technical skills across job postings to identify the most sought-after technologies, skill combinations, and salary trends.

The insights help job seekers prioritize learning paths while enabling recruiters to understand current market demand.

# Business Question 28
# Which skill domains are most in demand?

In [95]:
skill_demand = pd.read_sql("""
SELECT
    r.skill_domain,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.skill_domain
ORDER BY total_jobs DESC;
""", engine)

skill_demand

,skill_domain,total_jobs
0,Business Intelligence,11124
1,AI/ML/DL,6591
2,Cloud & DevOps,2344
3,Data Engineering,1595
4,Data Science,1547


# Business Question 29
# Which skill domains offer the highest average salary?

In [98]:
salary_by_skill = pd.read_sql("""
SELECT
    r.skill_domain,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
WHERE f.salary_disclosed = 1
GROUP BY r.skill_domain
HAVING COUNT(*) >= 20
ORDER BY avg_salary DESC;
""", engine)

salary_by_skill

,skill_domain,avg_salary,total_jobs
0,Cloud & DevOps,18.97,368
1,AI/ML/DL,18.24,785
2,Data Engineering,17.15,184
3,Data Science,13.76,204
4,Business Intelligence,12.34,1227


# Business Question 30
# Which skill domains require the highest average experience?

In [101]:
experience_by_skill = pd.read_sql("""
SELECT
    r.skill_domain,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_experience,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.skill_domain
HAVING COUNT(*) >= 20
ORDER BY avg_experience DESC;
""", engine)

experience_by_skill

,skill_domain,avg_experience,total_jobs
0,Cloud & DevOps,5.16,2344
1,Data Engineering,4.79,1595
2,AI/ML/DL,4.25,6591
3,Data Science,3.89,1547
4,Business Intelligence,3.79,11124


# Business Question 31
# Which work mode is most common within each skill domain?

In [104]:
workmode_skill = pd.read_sql("""
SELECT
    r.skill_domain,
    r.work_mode,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY
    r.skill_domain,
    r.work_mode
ORDER BY
    r.skill_domain,
    total_jobs DESC;
""", engine)

workmode_skill

,skill_domain,work_mode,total_jobs
0,AI/ML/DL,On-site,4943
1,AI/ML/DL,Remote,888
2,AI/ML/DL,Hybrid,760
3,Business Intelligence,On-site,9559
4,Business Intelligence,Remote,850
5,Business Intelligence,Hybrid,715
6,Cloud & DevOps,On-site,1658
7,Cloud & DevOps,Hybrid,538
8,Cloud & DevOps,Remote,148
9,Data Engineering,On-site,1238


# Business Question 32
# Which cities have the highest demand for each skill domain?

In [113]:
city_skill = pd.read_sql("""
SELECT
    l.primary_city,
    r.skill_domain,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE l.primary_city <> 'Remote'
GROUP BY
    l.primary_city,
    r.skill_domain
HAVING COUNT(*) >= 5
ORDER BY
    total_jobs DESC limit 10;
""", engine)

city_skill

,primary_city,skill_domain,total_jobs
0,Chennai,Business Intelligence,1340
1,Pune,Business Intelligence,1329
2,Mumbai,Business Intelligence,1321
3,Bangalore,Business Intelligence,1250
4,Gurgaon,Business Intelligence,1240
5,Noida,Business Intelligence,1234
6,Kolkata,Business Intelligence,904
7,Ahmedabad,Business Intelligence,861
8,Noida,AI/ML/DL,802
9,Mumbai,AI/ML/DL,783


## Section Summary

The Skills Intelligence analysis reveals hiring demand across different technology domains.

Key findings include:

- Most in-demand skill domains.
- Skill domains associated with higher salaries.
- Experience expectations by skill domain.
- Work mode preferences across domains.
- Geographic concentration of skill demand.

These insights help candidates align their learning roadmap with current market demand while supporting workforce planning and talent acquisition strategies.

## Section Summary

The Skills Intelligence analysis reveals hiring demand across different technology domains.

Key findings include:

- Most in-demand skill domains.
- Skill domains associated with higher salaries.
- Experience expectations by skill domain.
- Work mode preferences across domains.
- Geographic concentration of skill demand.

These insights help candidates align their learning roadmap with current market demand while supporting workforce planning and talent acquisition strategies.

# Business Question 33
# Which skill domains are most in demand?

In [123]:
skill_domain_demand = pd.read_sql("""
SELECT
    r.skill_domain,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.skill_domain
ORDER BY total_jobs DESC;
""", engine)

skill_domain_demand

,skill_domain,total_jobs
0,Business Intelligence,11124
1,AI/ML/DL,6591
2,Cloud & DevOps,2344
3,Data Engineering,1595
4,Data Science,1547


# Business Question 34
# Which skill domains offer the highest median salary?

In [126]:
salary_skill_df = pd.read_sql("""
SELECT
    r.skill_domain,
    f.salary_midpoint_lpa
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
WHERE f.salary_disclosed = 1;
""", engine)

salary_skill_summary = (
    salary_skill_df
    .groupby("skill_domain")["salary_midpoint_lpa"]
    .agg(
        median_salary="median",
        average_salary="mean",
        total_jobs="count"
    )
    .round(2)
    .sort_values("median_salary", ascending=False)
    .reset_index()
)

salary_skill_summary

,skill_domain,median_salary,average_salary,total_jobs
0,Cloud & DevOps,20.0,18.97,368
1,AI/ML/DL,15.0,18.24,785
2,Data Engineering,15.0,17.15,184
3,Data Science,14.0,13.76,204
4,Business Intelligence,12.5,12.34,1227


# Business Question 35
# Which skill domains require the highest average minimum experience?

In [129]:
experience_skill = pd.read_sql("""
SELECT
    r.skill_domain,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_min_experience,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.skill_domain
HAVING COUNT(*) >= 20
ORDER BY avg_min_experience DESC;
""", engine)

experience_skill

,skill_domain,avg_min_experience,total_jobs
0,Cloud & DevOps,5.16,2344
1,Data Engineering,4.79,1595
2,AI/ML/DL,4.25,6591
3,Data Science,3.89,1547
4,Business Intelligence,3.79,11124


# Business Question 36
# Which work mode is most common for each skill domain?

In [132]:
workmode_skill = pd.read_sql("""
SELECT
    r.skill_domain,
    r.work_mode,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY
    r.skill_domain,
    r.work_mode
ORDER BY
    r.skill_domain,
    total_jobs DESC;
""", engine)

workmode_skill

,skill_domain,work_mode,total_jobs
0,AI/ML/DL,On-site,4943
1,AI/ML/DL,Remote,888
2,AI/ML/DL,Hybrid,760
3,Business Intelligence,On-site,9559
4,Business Intelligence,Remote,850
5,Business Intelligence,Hybrid,715
6,Cloud & DevOps,On-site,1658
7,Cloud & DevOps,Hybrid,538
8,Cloud & DevOps,Remote,148
9,Data Engineering,On-site,1238


# Business Question 37
# Which cities have the highest demand for each skill domain?

In [137]:
city_skill = pd.read_sql("""
SELECT
    l.primary_city,
    r.skill_domain,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE l.primary_city <> 'Remote'
GROUP BY
    l.primary_city,
    r.skill_domain
HAVING COUNT(*) >= 5
ORDER BY
    total_jobs DESC limit 10;
""", engine)

city_skill

,primary_city,skill_domain,total_jobs
0,Chennai,Business Intelligence,1340
1,Pune,Business Intelligence,1329
2,Mumbai,Business Intelligence,1321
3,Bangalore,Business Intelligence,1250
4,Gurgaon,Business Intelligence,1240
5,Noida,Business Intelligence,1234
6,Kolkata,Business Intelligence,904
7,Ahmedabad,Business Intelligence,861
8,Noida,AI/ML/DL,802
9,Mumbai,AI/ML/DL,783


# Business Question 38
# Which skill domains have the highest salary growth potential?

In [140]:
skill_growth = pd.read_sql("""
SELECT
    r.skill_domain,
    ROUND(AVG(f.salary_midpoint_lpa),2) AS avg_salary,
    ROUND(AVG(f.experience_min_yrs),2) AS avg_experience
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
WHERE f.salary_disclosed = 1
GROUP BY r.skill_domain
HAVING COUNT(*) >= 20;
""", engine)

skill_growth["salary_per_year_experience"] = (
    skill_growth["avg_salary"] /
    skill_growth["avg_experience"].replace(0, 0.5)
).round(2)

skill_growth.sort_values(
    "salary_per_year_experience",
    ascending=False
)

,skill_domain,avg_salary,avg_experience,salary_per_year_experience
0,AI/ML/DL,18.24,4.53,4.03
1,Cloud & DevOps,18.97,5.59,3.39
3,Data Science,13.76,4.27,3.22
4,Data Engineering,17.15,5.53,3.10
2,Business Intelligence,12.34,4.03,3.06


# Recruitment Trends & Market Intelligence

## Objective

Analyze hiring trends across the Indian technology job market to understand recruitment patterns, hiring preferences, and market opportunities.

The analysis provides strategic insights for job seekers, recruiters, and business leaders by identifying emerging hiring patterns across roles, locations, work modes, and organizations.

# Business Question 39
# Which role categories dominate the Indian technology job market?

In [144]:
role_market_share = pd.read_sql("""
SELECT
    r.role_category,
    COUNT(*) AS total_jobs,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM fact_jobs),
        2
    ) AS market_share_percent
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY r.role_category
ORDER BY total_jobs DESC;
""", engine)

role_market_share

,role_category,total_jobs,market_share_percent
0,Data Scientist,6455,27.82
1,Data Analyst,4729,20.38
2,Business Analyst,4505,19.42
3,Machine Learning Engineer,4004,17.26
4,Data Engineer,1922,8.28
5,Python Developer,1586,6.84


Business Insight

Role categories with the highest market share represent the strongest hiring demand and indicate where candidates may find the greatest number of opportunities.

# Business Question 40
# Which companies are hiring across the widest variety of roles?

In [148]:
company_role_diversity = pd.read_sql("""
SELECT
    c.company_name,
    COUNT(DISTINCT r.role_category) AS unique_roles,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY c.company_name
HAVING COUNT(*) >= 10
ORDER BY unique_roles DESC,
         total_jobs DESC
LIMIT 20;
""", engine)

company_role_diversity

,company_name,unique_roles,total_jobs
0,Tata Consultancy Services,6,654
1,Accenture,6,438
2,EY,6,340
3,Leading Client,6,244
4,Infosys,6,209
5,Capgemini,6,195
6,Photon,6,193
7,Logic Planet,6,185
8,Hexaware Technologies,6,147
9,Sparix Global,6,126


# Business Question 41
# Which cities offer the widest variety of technology roles?

In [151]:
city_role_diversity = pd.read_sql("""
SELECT
    l.primary_city,
    COUNT(DISTINCT r.role_category) AS unique_roles,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_location l
ON f.location_key = l.location_key
JOIN dim_role r
ON f.role_key = r.role_key
WHERE l.primary_city <> 'Remote'
GROUP BY l.primary_city
HAVING COUNT(*) >= 20
ORDER BY unique_roles DESC,
         total_jobs DESC;
""", engine)

city_role_diversity

,primary_city,unique_roles,total_jobs
0,Mumbai,6,3276
1,Bangalore,6,2830
2,Chennai,6,2666
3,Pune,6,2606
4,Noida,6,2585
5,Gurgaon,6,2210
6,Kolkata,6,1735
7,Delhi,6,920
8,Hyderabad,6,538
9,Mumbai Suburban,6,33


# Business Question 42
# Which work mode is preferred across different role categories?

In [154]:
role_workmode = pd.read_sql("""
SELECT
    r.role_category,
    r.work_mode,
    COUNT(*) AS total_jobs
FROM fact_jobs f
JOIN dim_role r
ON f.role_key = r.role_key
GROUP BY
    r.role_category,
    r.work_mode
ORDER BY
    r.role_category,
    total_jobs DESC;
""", engine)

role_workmode

,role_category,work_mode,total_jobs
0,Business Analyst,On-site,3928
1,Business Analyst,Hybrid,325
2,Business Analyst,Remote,252
3,Data Analyst,On-site,4065
4,Data Analyst,Remote,381
5,Data Analyst,Hybrid,283
6,Data Engineer,On-site,1453
7,Data Engineer,Hybrid,273
8,Data Engineer,Remote,196
9,Data Scientist,On-site,4942


# Business Question 43
# Which companies have the highest salary disclosure rate?

Why this matters

Salary transparency is becoming increasingly important. This analysis identifies organizations that are more open about compensation.

In [158]:
salary_transparency = pd.read_sql("""
SELECT
    c.company_name,
    COUNT(*) AS total_jobs,
    SUM(f.salary_disclosed) AS disclosed_jobs,
    ROUND(
        SUM(f.salary_disclosed) * 100.0 / COUNT(*),
        2
    ) AS disclosure_rate
FROM fact_jobs f
JOIN dim_company c
ON f.company_key = c.company_key
GROUP BY c.company_name
HAVING COUNT(*) >= 10
ORDER BY disclosure_rate DESC,
         total_jobs DESC
LIMIT 20;
""", engine)

salary_transparency

,company_name,total_jobs,disclosed_jobs,disclosure_rate
0,US MNC (analytics),61,61.0,100.00
1,client of kaizen,20,20.0,100.00
2,Service based Top B2B MNC in IT Services Domain,14,14.0,100.00
3,MNC Group,12,12.0,100.00
4,Avance Consulting Services,10,10.0,100.00
5,Black and white Business Solution,14,13.0,92.86
6,ti Steps,14,12.0,85.71
7,orcapod consulting services private limited,11,9.0,81.82
8,KPI Partners,19,14.0,73.68
9,MNC,10,7.0,70.00


## Section Summary

The Recruitment Trends analysis provides a high-level view of the Indian technology job market.

Key findings include:

- Dominant technology roles in the market.
- Organizations hiring across diverse job categories.
- Cities offering the broadest career opportunities.
- Work mode preferences by role.
- Companies with the highest salary transparency.

These insights support career planning, workforce strategy, and hiring decision-making.